<b>Step 1: Environment Setup & Data Ingestion</b>

In [1]:
# Block 1: Environment Setup
# Apply Intel optimizations for scikit-learn to improve training times
from sklearnex import patch_sklearn
patch_sklearn()

import pandas as pd
import numpy as np

# Block 2: Data Ingestion
# Define the path to the raw data file assuming the standard project structure
file_path = '../data/raw/uber-raw-data-apr14.csv'

# Load a single month (April 2014) to establish the baseline pipeline
df = pd.read_csv(file_path)

# Display dataset dimensions and initial rows to verify ingestion
print(f"Dataset shape: {df.shape}\n")
print(df.info())
display(df.head())

Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


Dataset shape: (564516, 4)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 564516 entries, 0 to 564515
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   Date/Time  564516 non-null  object 
 1   Lat        564516 non-null  float64
 2   Lon        564516 non-null  float64
 3   Base       564516 non-null  object 
dtypes: float64(2), object(2)
memory usage: 17.2+ MB
None


,Date/Time,Lat,Lon,Base
0,4/1/2014 0:11:00,40.7690,-73.9549,B02512
1,4/1/2014 0:17:00,40.7267,-74.0345,B02512
2,4/1/2014 0:21:00,40.7316,-73.9873,B02512
3,4/1/2014 0:28:00,40.7588,-73.9776,B02512
4,4/1/2014 0:33:00,40.7594,-73.9722,B02512


<b>Step 2: Data Preprocessing & Feature Extraction

In [2]:
# Block 3: Data Preprocessing
# Convert the 'Date/Time' column from string object to datetime object
df['Date/Time'] = pd.to_datetime(df['Date/Time'], format='%m/%d/%Y %H:%M:%S')

# Extract specific temporal features required for segmentation
df['Hour'] = df['Date/Time'].dt.hour
df['DayOfWeek'] = df['Date/Time'].dt.day_name()
df['DayOfWeek_Num'] = df['Date/Time'].dt.dayofweek # 0=Monday, 6=Sunday for numerical sorting

# Drop missing values if any exist in the critical coordinate columns
df = df.dropna(subset=['Lat', 'Lon'])

# Verify the newly created features
display(df.head())

,Date/Time,Lat,Lon,Base,Hour,DayOfWeek,DayOfWeek_Num
0,2014-04-01 00:11:00,40.7690,-73.9549,B02512,0,Tuesday,1
1,2014-04-01 00:17:00,40.7267,-74.0345,B02512,0,Tuesday,1
2,2014-04-01 00:21:00,40.7316,-73.9873,B02512,0,Tuesday,1
3,2014-04-01 00:28:00,40.7588,-73.9776,B02512,0,Tuesday,1
4,2014-04-01 00:33:00,40.7594,-73.9722,B02512,0,Tuesday,1


<b>Step 3: Baseline Clustering (KMeans) </b>
<br> filtering data to a specific temporal window before applying the algorithm.

In [9]:
# Block 4: Baseline Clustering (KMeans)
from sklearn.cluster import KMeans

# Isolate the dataset to a specific time window to establish the baseline
# Example: Tuesdays at 17:00 (5 PM rush hour)
mask = (df['DayOfWeek'] == 'Tuesday') & (df['Hour'] == 17)
df_subset = df[mask].copy()

# Extract geographical coordinates for the model
X = df_subset[['Lat', 'Lon']]

# Initialize and fit the KMeans algorithm
# Note: n_clusters=8 is chosen as an arbitrary starting point
kmeans = KMeans(n_clusters=8, random_state=42, n_init='auto')
df_subset['Cluster'] = kmeans.fit_predict(X)
# Convert Cluster to string/category so Plotly treats it as discrete colors later
df_subset['Cluster'] = df_subset['Cluster'].astype(str)

print(f"Number of trips in the filtered subset: {len(df_subset)}")
display(df_subset.head())

Number of trips in the filtered subset: 8297


,Date/Time,Lat,Lon,Base,Hour,DayOfWeek,DayOfWeek_Num,Cluster
634,2014-04-01 17:00:00,40.7591,-73.9670,B02512,17,Tuesday,1,7
635,2014-04-01 17:00:00,40.7701,-73.9625,B02512,17,Tuesday,1,1
636,2014-04-01 17:02:00,40.7789,-73.9559,B02512,17,Tuesday,1,1
637,2014-04-01 17:02:00,40.7789,-73.9559,B02512,17,Tuesday,1,1
638,2014-04-01 17:02:00,40.7330,-73.9824,B02512,17,Tuesday,1,5


<b>Step 4: Spatial Visualization </b>
<br>Data points projected onto an interactive map to visually validate the logical grouping of the KMeans algorithm.

In [10]:
# Block 5: Spatial Visualization
import plotly.express as px

# Generate an interactive mapbox scatter plot
fig = px.scatter_mapbox(
    df_subset,
    lat="Lat",
    lon="Lon",
    color="Cluster",
    zoom=10,
    mapbox_style="carto-positron", 
    title="Uber Pickups Hot-Zones: Tuesdays at 17:00 (KMeans, k=8)",
    width=900,
    height=600
)

# Adjust marker properties for better visibility over dense urban areas
fig.update_traces(marker=dict(size=4, opacity=0.6))

# Display the map inline
fig.show()

# Export the visualization to the assets folder for final deliverables
# Ensure the directory exists before saving
import os
os.makedirs('../assets', exist_ok=True)
fig.write_html('../assets/kmeans_tuesday_17h.html')

C:\Users\ENava\AppData\Local\Temp\ipykernel_23988\10475313.py:5: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



<b>Step 5: Algorithm Comparison (DBSCAN)

In [11]:
# Block 6: Algorithm Comparison - DBSCAN
from sklearn.cluster import DBSCAN

# Initialize DBSCAN
# Note: eps=0.005 roughly translates to ~500 meters in latitude/longitude degrees.
# min_samples=15 enforces that a valid 'hot-zone' must have at least 15 pickups in that radius.
dbscan = DBSCAN(eps=0.005, min_samples=15)

# Fit the algorithm and assign labels
df_subset['Cluster_DBSCAN'] = dbscan.fit_predict(X)

# Isolate noise points (-1) from valid clusters for reporting
n_clusters_ = len(set(dbscan.labels_)) - (1 if -1 in dbscan.labels_ else 0)
n_noise_ = list(dbscan.labels_).count(-1)

print(f"Estimated number of DBSCAN clusters: {n_clusters_}")
print(f"Estimated number of noise points: {n_noise_} out of {len(df_subset)} total points\n")

# Convert labels to strings for discrete categorical coloring in Plotly
df_subset['Cluster_DBSCAN'] = df_subset['Cluster_DBSCAN'].astype(str)

# Generate an interactive mapbox scatter plot for DBSCAN
import plotly.express as px

fig_dbscan = px.scatter_mapbox(
    df_subset,
    lat="Lat",
    lon="Lon",
    color="Cluster_DBSCAN",
    zoom=10,
    mapbox_style="carto-positron", 
    title="Uber Pickups Hot-Zones: Tuesdays at 17:00 (DBSCAN)",
    width=900,
    height=600
)

# Adjust marker properties
fig_dbscan.update_traces(marker=dict(size=4, opacity=0.6))

# Display and export the map
fig_dbscan.show()
fig_dbscan.write_html('../assets/dbscan_tuesday_17h.html')

Estimated number of DBSCAN clusters: 8
Estimated number of noise points: 527 out of 8297 total points



C:\Users\ENava\AppData\Local\Temp\ipykernel_23988\2122812819.py:25: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

